# Research Version 3 ? 05: Templates and text robustness

Read exact/template candidate examples with source support. Similarity is an audit signal, not proof of leakage. Distinguish sampled coverage from a complete corpus claim.

This notebook runs only in Colab and Google Drive. No Windows checkout, local datasets, local model files or replication-manifest file are required. Use the new experiment ID; do not mix this package with an older cpu-v3 archive.


## Scope and compatibility patch

Text robustness for both text datasets; text normalization helpers come from dataset-independent text_utils.


# 05: Template, near-duplicate and source-sensitivity research

## 1. Purpose

Investigate templates, near-duplicate candidates and source/event sensitivity without changing partitions.

## 2. Prerequisites and outputs

**Required inputs:** Notebook 04 baseline predictions and earlier prepared text.

**This stage saves:** Five-task audits, candidate CSV files and robustness summary/report.

Required earlier notebook markers: 04.

All runtime files belong to this Colab/Drive experiment; no external project files are read.

## 3. Configuration

Use the same experiment ID/folder throughout. All action switches default False. Enable Drive/bootstrap and RUN_STAGE deliberately; enable installation on a fresh runtime. Notebook 01 needs dataset download permission; notebook 08 needs pretrained download permission.

In [ ]:
EXPERIMENT_ID = "colab-cpu-v3-clean"
DRIVE_FOLDER = "ChainShield-Colab"
ENABLE_DRIVE = False
BOOTSTRAP_EXPERIMENT = False
ALLOW_PACKAGE_INSTALL = False
ALLOW_DATA_DOWNLOADS = False
ALLOW_PRETRAINED_DOWNLOADS = False
RUN_STAGE = False
SYNC_INTERVAL_SECONDS = 60
VERIFY_PERSISTENCE_FULL = False
VERIFY_CHECKPOINT_PAIRS = False
CLEAR_STALE_WRITER_LOCK = False
CONFIRM_NO_OTHER_WRITER = False
DOWNLOAD_REPORT_ZIP = False

# Isolated plotting workers must not inherit the Colab inline backend.
if RUN_STAGE:
    import os
    os.environ["MPLBACKEND"] = "Agg"


## 4. Readable stage implementation

The small cell command below registers visible module text in memory. It does not execute the module. This notebook supplies only the files listed here; other supporting modules are restored from earlier Colab stages.

In [ ]:
# This small cell registers the visible stage code; it writes no files.
REGISTERED_FILES = {}
def register_stage_file(line, body):
    name=line.strip()
    if not name or name.startswith("/") or "\\" in name or ":" in name or ".." in name.split("/"):
        raise ValueError("Use a safe relative module name")
    REGISTERED_FILES[name]=body.rstrip("\n")+"\n"
get_ipython().register_magic_function(register_stage_file,"cell","chainshield_file")


### src/chainshield/data/robustness.py


In [ ]:
%%chainshield_file src/chainshield/data/robustness.py
"""Bounded train/development text diagnostics; candidates are not deduplication decisions."""
from __future__ import annotations
from collections import Counter,defaultdict
import hashlib
import re
import numpy as np
from sklearn import config_context
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score,classification_report,f1_score
from sklearn.neighbors import NearestNeighbors
from .text_utils import clean_text,text_key


def template_signature(text,minimum_lexical_tokens=5):
    """Normalize mechanical variation, preserving all lexical place/event words."""
    value=clean_text(text).casefold()
    value=re.sub(r"^rt\s+","",value)
    value=re.sub(r"https?://\S+|www\.\S+"," <url> ",value)
    value=re.sub(r"(?<!\w)@\w+"," <user> ",value)
    value=re.sub(r"(?<!\w)\d+(?:[.,]\d+)*(?!\w)"," <number> ",value)
    value=" ".join(value.split())
    lexical=re.findall(r"[^\W\d_]+",re.sub(r"<(?:url|user|number)>","",value),flags=re.UNICODE)
    return hashlib.sha256(value.encode("utf-8")).hexdigest() if len(lexical)>=minimum_lexical_tokens else ""


def deterministic_sample_indexes(identifiers,maximum=1500,seed=42):
    if maximum<1 or len(identifiers)!=len(set(identifiers)):
        raise ValueError("A positive cap and unique record IDs are required")
    ranked=sorted(range(len(identifiers)),
                  key=lambda i:hashlib.sha256(f"{seed}:{identifiers[i]}".encode()).hexdigest())
    return ranked[:maximum]


def template_candidates(train,dev):
    groups=defaultdict(list)
    for i,text in enumerate(train["texts"]):
        signature=template_signature(text)
        if signature:
            groups[signature].append(i)
    candidates=[]
    eligible_dev=0
    for i,text in enumerate(dev["texts"]):
        signature=template_signature(text)
        if signature:
            eligible_dev+=1
        matches=groups.get(signature,[])
        if not matches:
            continue
        # Exact matches are reported separately; this detector targets mechanical variants.
        variants=[j for j in matches if text_key(train["texts"][j])!=text_key(text)]
        if variants:
            first=min(variants,key=lambda j:train["ids"][j])
            candidates.append({
                "dev_record_id":dev["ids"][i],"train_record_id":train["ids"][first],
                "signature_sha256":signature,"training_group_size":len(matches),
                "distinct_text_training_matches":len(variants),
                "dev_label":dev["labels"][i],"train_label":train["labels"][first],
                "labels_agree":dev["labels"][i]==train["labels"][first],
            })
    return candidates,{
        "training_template_eligible_rows":sum(len(v) for v in groups.values()),
        "training_signature_groups":len(groups),
        "training_repeated_signature_groups":sum(len(v)>1 for v in groups.values()),
        "training_rows_in_repeated_signature_groups":sum(len(v) for v in groups.values() if len(v)>1),
        "dev_template_eligible_rows":eligible_dev,"dev_candidate_rows":len(candidates),
        "candidate_fraction_all_dev":len(candidates)/len(dev["ids"]) if dev["ids"] else None,
    }


def near_candidates(train,dev,*,maximum_queries=1500,seed=42,threshold=0.9,
                    maximum_features=30000,batch_size=32,maximum_chars=2000):
    """Training-fitted char TF-IDF; nearest query batches bound distance-matrix size."""
    if not train["texts"] or not dev["texts"] or not 0<=threshold<=1:
        raise ValueError("Nonempty partitions and a valid threshold are required")
    if batch_size<1 or maximum_features<1 or maximum_chars<1:
        raise ValueError("Resource bounds must be positive")
    selected=deterministic_sample_indexes(dev["ids"],maximum_queries,seed)
    vectorizer=TfidfVectorizer(analyzer="char_wb",ngram_range=(3,5),min_df=2,
                              max_features=maximum_features,dtype=np.float32,norm="l2")
    training=[text[:maximum_chars] for text in train["texts"]]
    queries=[dev["texts"][i][:maximum_chars] for i in selected]
    try:
        matrix=vectorizer.fit_transform(training)
    except ValueError as error:
        raise ValueError("Character vocabulary could not be fitted on training data") from error
    engine=NearestNeighbors(n_neighbors=1,metric="cosine",algorithm="brute",n_jobs=2).fit(matrix)
    neighbors=[]
    with config_context(working_memory=128):
        for start in range(0,len(selected),batch_size):
            query=vectorizer.transform(queries[start:start+batch_size])
            distances,indexes=engine.kneighbors(query)
            for offset,(distance,index) in enumerate(zip(distances[:,0],indexes[:,0])):
                i=selected[start+offset]
                j=int(index)
                similarity=float(np.clip(1-float(distance),0,1))
                neighbors.append({
                    "dev_record_id":dev["ids"][i],"train_record_id":train["ids"][j],
                    "cosine_similarity":similarity,"candidate":similarity>=threshold,
                    "exact_normalized_text":text_key(dev["texts"][i])==text_key(train["texts"][j]),
                    "dev_label":dev["labels"][i],"train_label":train["labels"][j],
                    "labels_agree":dev["labels"][i]==train["labels"][j],
                })
    similarities=np.array([row["cosine_similarity"] for row in neighbors])
    candidates=[row for row in neighbors if row["candidate"]]
    summary={
        "method":"training-fitted char_wb TF-IDF 3-5 grams, cosine nearest neighbor",
        "threshold":threshold,"seed":seed,"query_cap":maximum_queries,
        "query_count":len(neighbors),"training_rows":len(train["ids"]),
        "candidate_rows":len(candidates),
        "candidate_fraction_sample":len(candidates)/len(neighbors),
        "exact_match_rows_sample":sum(row["exact_normalized_text"] for row in neighbors),
        "nearest_cosine_median":float(np.median(similarities)),
        "nearest_cosine_p95":float(np.percentile(similarities,95)),
        "vocabulary_features":len(vectorizer.vocabulary_),
        "maximum_features":maximum_features,"batch_size":batch_size,"maximum_chars":maximum_chars,
        "training_rows_truncated":sum(len(t)>maximum_chars for t in train["texts"]),
        "query_rows_truncated":sum(len(t)>maximum_chars for t in
                                    (dev["texts"][i] for i in selected)),
        "csr_matrix_storage_bytes":matrix.data.nbytes+matrix.indices.nbytes+matrix.indptr.nbytes,
        "storage_note":"CSR storage only; excludes vocabulary, solver workspace, and process memory",
        "sampling_note":"Deterministic hash sample; candidate rate is diagnostic, not an independent-generalization estimate",
    }
    return neighbors,summary


def prediction_metrics(predictions,classes):
    if not predictions:
        return {"records":0,"accuracy":None,"macro_f1_global_classes":None,
                "macro_f1_supported_classes":None,"label_counts":{}}
    actual=[row["actual_label"] for row in predictions]
    predicted=[row["predicted_label"] for row in predictions]
    supported=sorted(set(actual))
    return {
        "records":len(predictions),"accuracy":float(accuracy_score(actual,predicted)),
        "macro_f1_global_classes":float(f1_score(actual,predicted,labels=classes,
                                                average="macro",zero_division=0)),
        "macro_f1_supported_classes":float(f1_score(actual,predicted,labels=supported,
                                                   average="macro",zero_division=0)),
        "label_counts":dict(Counter(actual)),
        "classification_report":classification_report(actual,predicted,labels=classes,
                                                      output_dict=True,zero_division=0),
        "interpretation":"Saved validation predictions only; no model retraining or domain-held-out claim",
    }


def grouped_prediction_metrics(predictions,metadata,field,classes):
    groups=defaultdict(list)
    for row in predictions:
        if row["record_id"] not in metadata:
            raise ValueError("Prediction record is missing source metadata")
        groups[str(metadata[row["record_id"]].get(field,"UNKNOWN"))].append(row)
    return {name:prediction_metrics(rows,classes) for name,rows in sorted(groups.items())}


def subset_prediction_metrics(predictions,template_rows,neighbors,classes):
    template_ids={r["dev_record_id"] for r in template_rows}
    sampled_ids={r["dev_record_id"] for r in neighbors}
    near_ids={r["dev_record_id"] for r in neighbors if r["candidate"]}
    subsets={
        "all_validation":predictions,
        "template_candidates":[r for r in predictions if r["record_id"] in template_ids],
        "no_template_candidate_detected":[r for r in predictions if r["record_id"] not in template_ids],
        "sampled_near_candidates":[r for r in predictions if r["record_id"] in near_ids],
        "sampled_no_near_candidate_detected":[r for r in predictions
                 if r["record_id"] in sampled_ids-near_ids],
        "sampled_neither_detector_flagged":[r for r in predictions
                 if r["record_id"] in sampled_ids-near_ids-template_ids],
    }
    return {name:prediction_metrics(rows,classes) for name,rows in subsets.items()}


### scripts/audit_text_robustness.py


In [ ]:
%%chainshield_file scripts/audit_text_robustness.py
"""Audit text baselines using only approved training/development data."""
from __future__ import annotations
import csv
from datetime import datetime,timezone
import json
from pathlib import Path
import sys
from time import perf_counter

ROOT=Path(__file__).resolve().parents[1]
sys.path.insert(0,str(ROOT/"src"))
from threadpoolctl import threadpool_limits
from chainshield.data.robustness import (
    template_candidates,near_candidates,grouped_prediction_metrics,subset_prediction_metrics,
)
from chainshield.ml.train import sha256
from chainshield.nlp.baseline import load_text_partitions

POLICY={"seed":42,"maximum_queries":1500,"threshold":0.9,
        "maximum_features":30000,"batch_size":32,"maximum_chars":2000}


def source_metadata(directory,dataset,partition):
    selected={identifier for split in partition.values() for identifier in split["ids"]}
    result={}
    identity="tweet_id" if dataset=="crisismmd" else "record_id"
    with (directory/"canonical.csv").open(encoding="utf-8",newline="") as handle:
        for row in csv.DictReader(handle):
            identifier=row[identity]
            if identifier not in selected:
                continue
            result[identifier]=(
                {"event_name":row["event_name"]} if dataset=="crisismmd" else
                {"collection":row["collection"],"source_file":row["source_file"]})
    if set(result)!=selected:
        raise ValueError("Selected source metadata is incomplete")
    return result


def saved_predictions(result,dev):
    directory=Path(result["artifact"]).resolve()
    directory.relative_to(ROOT/"artifacts/baselines")
    metadata=result["metadata"]
    if sha256(directory/"model.joblib")!=metadata["artifact_sha256"]:
        raise ValueError("Baseline model integrity mismatch")
    path=directory/"dev_predictions.csv"
    if sha256(path)!=metadata["prediction_files"]["dev_predictions.csv"]:
        raise ValueError("Baseline prediction integrity mismatch")
    with path.open(encoding="utf-8",newline="") as handle:
        rows=list(csv.DictReader(handle))
    expected=dict(zip(dev["ids"],dev["labels"]))
    if len(rows)!=len(expected) or {r["record_id"] for r in rows}!=set(expected):
        raise ValueError("Validation predictions do not match selected task/view records")
    if any(r["partition"]!="dev" or r["actual_label"]!=str(expected[r["record_id"]]) for r in rows):
        raise ValueError("Prediction partition or target mismatch")
    return rows


def write_csv(path,rows,fields):
    with path.open("x",encoding="utf-8",newline="") as handle:
        writer=csv.DictWriter(handle,fieldnames=fields)
        writer.writeheader()
        writer.writerows(rows)


def report_markdown(reports):
    lines=["# Text robustness audit","",
        "Diagnostic audit of existing training and validation partitions. No test evaluation, retraining, or dataset edits were performed.",
        "Candidates identify mechanical resemblance and require review; they do not establish leakage or semantic equivalence.","",
        "## Method","",
        "- Character TF-IDF: 3-5 character grams within word boundaries, vocabulary fitted only on training text.",
        "- At most 1,500 deterministic hash-sampled validation queries per task, cosine threshold 0.90.",
        "- All selected training texts are searched in batches of 32; vocabulary cap 30,000; character cap 2,000.",
        "- Template signatures normalize URLs, user mentions, numeric values, whitespace, and a leading retweet marker.",
        "- Place names, disaster terms, and lexical content are preserved; signatures need at least five lexical tokens.",
        "- Source/event metrics are breakdowns of existing validation predictions, not held-out-domain experiments.",
        "- No raw tweet text is copied into candidate CSVs.","",
        "## Summary","",
        "| Dataset / task | Dev rows | Template candidates | Near candidates / queried | Overall macro-F1 | Unflagged sampled macro-F1 |",
        "|---|---:|---:|---:|---:|---:|"]
    for report in reports:
        subset=report["subsets"]
        overall=subset["all_validation"]
        unflagged=subset["sampled_neither_detector_flagged"]
        value=unflagged["macro_f1_global_classes"]
        formatted=f"{value:.4f}" if value is not None else "-"
        near=report["near_duplicate_summary"]
        lines.append(f"| {report['dataset']} / {report['task']} | {overall['records']} | "
            f"{report['template_summary']['dev_candidate_rows']} | "
            f"{near['candidate_rows']} / {near['query_count']} | "
            f"{overall['macro_f1_global_classes']:.4f} | {formatted} |")
    for report in reports:
        lines+=["","## "+report["dataset"]+" / "+report["task"],"",
            f"Validation view: {report['view']}. Train rows: {report['training_rows']}; validation rows: {report['validation_rows']}.",
            f"Exact normalized train/validation overlaps: {report['exact_text_overlap_groups']}.",
            f"Near-neighbor median cosine: {report['near_duplicate_summary']['nearest_cosine_median']:.4f}; "
            f"p95: {report['near_duplicate_summary']['nearest_cosine_p95']:.4f}.",""]
        for field,groups in report["source_breakdowns"].items():
            lines += ["### Validation breakdown by "+field,"",
                "| Group | Train records | Dev records | Accuracy | Macro-F1 (supported classes) | Macro-F1 (global classes) |",
                "|---|---:|---:|---:|---:|---:|"]
            counts=report["training_source_counts"][field]
            for group,metrics in groups.items():
                lines.append(f"| {group.replace('|','/')} | {counts.get(group,0)} | {metrics['records']} | "
                    f"{metrics['accuracy']:.4f} | {metrics['macro_f1_supported_classes']:.4f} | "
                    f"{metrics['macro_f1_global_classes']:.4f} |")
        lines+=["","### Candidate-sensitive validation subsets","",
            "| Subset | Records | Accuracy | Macro-F1 (global classes) |",
            "|---|---:|---:|---:|"]
        for name,metrics in report["subsets"].items():
            accuracy=f"{metrics['accuracy']:.4f}" if metrics["accuracy"] is not None else "-"
            macro=f"{metrics['macro_f1_global_classes']:.4f}" if metrics["macro_f1_global_classes"] is not None else "-"
            lines.append(f"| {name} | {metrics['records']} | {accuracy} | {macro} |")
    lines+=["","## Interpretation and limitations","",
        "High near-neighbor similarity can arise from topical vocabulary, shared hashtags, quotations, retweets, or source templates. Numerical changes may change factual meaning.",
        "A low candidate rate does not establish semantic independence. The nearest-neighbor audit samples validation queries and has a bounded vocabulary.",
        "The unflagged sampled subset is not a new clean benchmark: other duplicates may remain and class/source composition differs from the full validation set.",
        "Global-class macro-F1 includes zero-support classes; supported-class macro-F1 includes only labels present in that subgroup's true targets. Compare with class counts in audit.json.",
        "Source and event breakdowns use a model trained on the existing mixed training set. They cannot demonstrate transfer to a new source or unseen event.",
        "Source-authenticity concerns and rounded source IDs remain documented limitations; similarity alone does not establish synthetic data.",
        "Any future exclusion policy must be reviewed and versioned before generating new splits or selecting models. Existing datasets and models were not changed.",
        "Final test partitions remain unused. A separate, predeclared held-out-domain/event experiment is still needed for stronger generalization evidence.",""]
    return "\n".join(lines)


def main():
    destination=ROOT/"artifacts/baselines/robustness"
    report_path=ROOT/"docs/text-robustness.md"
    if destination.exists() or report_path.exists():
        raise FileExistsError("Refusing to overwrite an existing robustness audit")
    summary_path=ROOT/"artifacts/baselines/baseline_summary.json"
    baseline=json.loads(summary_path.read_text(encoding="utf-8"))
    results=[r for r in baseline["results"] if r["metadata"]["dataset"] in {"india_nepal","crisismmd"}]
    reports=[]
    outputs=[]
    with threadpool_limits(limits=2):
        for result in results:
            metadata=result["metadata"]
            dataset,task=metadata["dataset"],metadata["task"]
            print("Auditing "+dataset+"/"+task,flush=True)
            started=perf_counter()
            directory=ROOT/"data/processed/research-v2/paired_v2/control"/dataset
            partition=load_text_partitions(directory,dataset,task,
                metadata["view"] if dataset=="crisismmd" else None)
            train,dev=partition["train"],partition["dev"]
            sources=source_metadata(directory,dataset,partition)
            predictions=saved_predictions(result,dev)
            classes=[str(c) for c in metadata["classes"]]
            templates,template_summary=template_candidates(train,dev)
            print(f"  Template candidates: {len(templates)} / {len(dev['ids'])}",flush=True)
            neighbors,near_summary=near_candidates(train,dev,**POLICY)
            print(f"  Near candidates: {near_summary['candidate_rows']} / {near_summary['query_count']}",flush=True)
            fields=["event_name"] if dataset=="crisismmd" else ["collection","source_file"]
            source_counts={field:{} for field in fields}
            for field in fields:
                for identifier in train["ids"]:
                    group=sources[identifier][field]
                    source_counts[field][group]=source_counts[field].get(group,0)+1
            # Check exact normalized text independently of candidate thresholds.
            from chainshield.data.text_utils import text_key
            overlap={text_key(text) for text in train["texts"]}&{text_key(text) for text in dev["texts"]}
            audit={
                "dataset":dataset,"task":task,"view":metadata["view"],
                "training_rows":len(train["ids"]),"validation_rows":len(dev["ids"]),
                "baseline_model_version":metadata["model_version"],
                "baseline_predictions_sha256":metadata["prediction_files"]["dev_predictions.csv"],
                "source_quality_sha256":sha256(directory/"quality.json"),
                "exact_text_overlap_groups":len(overlap),
                "template_summary":template_summary,"near_duplicate_summary":near_summary,
                "training_source_counts":source_counts,
                "source_breakdowns":{field:grouped_prediction_metrics(predictions,sources,field,classes)
                                     for field in fields},
                "subsets":subset_prediction_metrics(predictions,templates,neighbors,classes),
                "audit_seconds":perf_counter()-started,"test_evaluated":False,
            }
            if overlap:
                raise ValueError("Unexpected exact-text overlap; report rather than silently continue")
            reports.append(audit)
            outputs.append((dataset,task,templates,neighbors,audit))
    # Persist only after every task has passed its source and prediction checks.
    template_fields=["dev_record_id","train_record_id","signature_sha256","training_group_size",
                     "distinct_text_training_matches","dev_label","train_label","labels_agree"]
    near_fields=["dev_record_id","train_record_id","cosine_similarity","candidate",
                 "exact_normalized_text","dev_label","train_label","labels_agree"]
    for dataset,task,templates,neighbors,audit in outputs:
        directory=destination/dataset/task
        directory.mkdir(parents=True,exist_ok=True)
        write_csv(directory/"template_candidates.csv",templates,template_fields)
        write_csv(directory/"nearest_neighbors.csv",neighbors,near_fields)
        audit["output_sha256"]={name:sha256(directory/name) for name in
                               ("template_candidates.csv","nearest_neighbors.csv")}
        with (directory/"audit.json").open("x",encoding="utf-8") as handle:
            json.dump(audit,handle,indent=2,allow_nan=False)
            handle.write("\n")
    summary={"schema_version":1,"created_at_utc":datetime.now(timezone.utc).isoformat(),
        "baseline_summary_sha256":sha256(summary_path),"policy":POLICY,
        "test_evaluated":False,"datasets_changed":False,"models_retrained":False,
        "code_sha256":{"src/chainshield/data/robustness.py":sha256(ROOT/"src/chainshield/data/robustness.py"),
                       "scripts/audit_text_robustness.py":sha256(Path(__file__))},
        "results":reports}
    with (destination/"summary.json").open("x",encoding="utf-8") as handle:
        json.dump(summary,handle,indent=2,allow_nan=False)
        handle.write("\n")
    with report_path.open("x",encoding="utf-8") as handle:
        handle.write(report_markdown(reports))
    print("Audit complete for "+str(len(reports))+" text baselines; existing artifacts unchanged.",flush=True)


if __name__=="__main__":
    main()


In [ ]:
import hashlib,json
EXPECTED_STAGE_FILES={'src/chainshield/data/robustness.py': '826c9eaaae9df08f6bbc3c3be369f47212ae92f7fa2532ba2fca8cb29a3e786e', 'scripts/audit_text_robustness.py': '9e5de7556d1ead430643b8d60c7e4a90b078b57ddda95ed6036774467a8e3416'}
assert set(REGISTERED_FILES)==set(EXPECTED_STAGE_FILES)
for name,expected in EXPECTED_STAGE_FILES.items():
    if hashlib.sha256(REGISTERED_FILES[name].encode()).hexdigest()!=expected:raise ValueError("Changed source: "+name)


## 5. Attach, verify prerequisites and execute

Bootstrap connects Drive and restores committed work. Source installation commits only this stage definitions. Dependencies must already have completed; this notebook does not silently train them.

In [ ]:
# Connect the Colab experiment and load the shared runtime.
from pathlib import Path
CONTROLLER_SHA = '8c092c59baffc0804aeca86b4f0ac3fb6d4be3cd96eea96c9e7299e559d9c9e6'
IMPLEMENTATION_SHA256 = CONTROLLER_SHA
rt = None
if BOOTSTRAP_EXPERIMENT:
    if not ENABLE_DRIVE:
        raise PermissionError("Set ENABLE_DRIVE=True")
    from google.colab import drive
    drive.mount("/content/drive")
    runtime_path=Path("/content/drive/MyDrive")/DRIVE_FOLDER/"experiments"/EXPERIMENT_ID/"workspace/.replication/controller.py"
    if not runtime_path.is_file():
        raise RuntimeError("Complete Version 3 notebook 00 first; its shared controller is missing")
    runtime_code=runtime_path.read_text(encoding="utf-8")
    if hashlib.sha256(runtime_code.encode()).hexdigest()!=CONTROLLER_SHA:
        raise ValueError("Shared runtime version differs; do not mix notebook packages")
    exec(compile(runtime_code,"<Colab shared runtime>","exec"),globals())
    rt=Replica(EXPERIMENT_ID,drive_base=DRIVE_FOLDER,allow_drive=True)
    if CLEAR_STALE_WRITER_LOCK:
        rt.clear_stale_lock(acknowledge_no_other_writer=CONFIRM_NO_OTHER_WRITER)
else:
    print("Bootstrap disabled: no Drive access or filesystem changes")


In [ ]:
if RUN_STAGE:
    if rt is None:
        raise RuntimeError("Enable Drive/bootstrap first")
    rt.require_stage('04')
    rt.install_sources(REGISTERED_FILES)
    rt.environment(allow_install=ALLOW_PACKAGE_INSTALL)
else:
    print("Stage execution disabled")


### Audit repeated and similar text

This action executes only with RUN_STAGE=True. Progress and worker resource measurements are written to the experiment run directory.

In [ ]:
if RUN_STAGE:
    if not (rt.root/"artifacts/baselines/robustness/summary.json").exists():
        rt.execute("05-robustness",["scripts/audit_text_robustness.py"],SYNC_INTERVAL_SECONDS)
    else:
        print("Reusing committed diagnostic audit")
else:
    print("Stage execution disabled")


## 6. Results and interpretation

Similarity candidates require interpretation; do not silently delete them or claim they prove leakage.

The following cell reads this experiment outputs; definitions alone are not evidence that a stage ran.

In [ ]:
if rt is None:
    print("No attached experiment")
else:
    path=rt.root/"artifacts/baselines/robustness/summary.json"
    if path.exists(): print(json.dumps(read_json(path),indent=2))


## 7. Persist and hand over

Only after the preceding actions succeed, run the following cell to synchronize verified outputs and record this notebook completion. After an error, stop; repair/restart and rerun in order before recording completion. The next stage reads saved artifacts/components from Drive.

In [ ]:
if RUN_STAGE:
    rt.finish_stage('05')
else:
    print("Stage execution disabled")


## Recovery and boundaries

Run one writer per experiment. After interruption, disconnect and delete the Colab runtime, reconnect and restore this same experiment. Completed models are verified/reused; neural checkpoints resume compatible state. Unsynchronized work may repeat. Do not clear a writer lock until no other session is active.

No final-test fitting/scoring/tuning is performed. Model outputs do not verify incident truth or shipment exposure. Hosted Colab execution remains unverified until exercised in your session.